# 02 · Check source files

Inspects approved local inputs and records SHA-256 digests. The technology source tables are optional for the short accepted-evidence replay.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
#!/usr/bin/env python3
"""Check source and accepted-evidence inputs without writing scientific data."""
from __future__ import annotations
import argparse
import hashlib
import json
from pathlib import Path
import pandas as pd

INPUTS = {
    'jagermeyr': 'technology/01_global_irrigation_system_source_jagermeyr2015.csv',
    'rohwer': 'technology/02_global_rohwer_irrigation_reference.csv',
    'net_master': 'crop/01_FABLE_crop_water_requirement_master_2000_2020_FINAL.csv.gz',
    'approved_shares': 'technology/12_FABLE_irrigation_technology_shares_2000_2020_FINAL.csv',
    'efficiencies': 'technology/13_FABLE_irrigation_efficiency_parameters_FINAL.csv',
}


def locate(root: Path, rel: str) -> Path:
    path = root / rel
    alternatives = [path]
    if path.name.endswith('.csv.gz'):
        alternatives.append(path.with_suffix(''))
    elif path.name.endswith('.csv'):
        alternatives.append(Path(str(path)+'.gz'))
    return next((p for p in alternatives if p.is_file()), path)


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for part in iter(lambda: f.read(4*1024*1024), b''):
            h.update(part)
    return h.hexdigest()


def inspect(root: Path) -> dict:
    records = {}
    for key, relative in INPUTS.items():
        path = locate(root, relative)
        rec = {'path': str(path), 'available': path.is_file()}
        if path.is_file():
            rec['sha256'] = digest(path)
            rec['columns'] = list(pd.read_csv(path, nrows=0).columns)
        records[key] = rec
    return records


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
DATA_ROOT = DATA


## Execute and inspect

In [ ]:
records = inspect(DATA_ROOT)
print(json.dumps(records, indent=2))
missing = [name for name in ('net_master', 'approved_shares', 'efficiencies')
           if not records[name]['available']]
if missing:
    raise FileNotFoundError(f'Missing required accepted inputs: {missing}')
print('STATUS: PASS_IWR_ACCEPTED_EVIDENCE_PREFLIGHT_ONLY')
